# hparam-precedence-merge — worked example 3: Config merge filtering sentinel None from CLI layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `hparam-precedence-merge`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Argument parsers like `argparse` use `None` as the sentinel for flags that the user did NOT provide. During config merging, these sentinel `None` values should NOT override values set in the lower-priority config file — they represent absence, not an explicit choice of `None`. The fix: filter `cli_args` to remove `None` values before merging, so only explicitly set CLI arguments participate in the override.

## Worked solution

**Step 1 — create a realistic argparse-style CLI dict.** Keys for flags the user set have real values; keys for unset flags have `None`.

**Step 2 — filter out the Nones.** `cli_filtered = {k: v for k, v in cli_args.items() if v is not None}`. This removes unset flags.

**Step 3 — merge with precedence.** Apply `{**defaults, **file_cfg, **cli_filtered}`. The filtered CLI still overrides the file for keys the user explicitly set.

**Step 4 — verify.** A key the user set in CLI wins over the file value. A key the user did NOT set (None in CLI) keeps the file value. A key absent from CLI entirely also keeps the file value.

In [ ]:
import torch as t

t.manual_seed(0)

def merge_with_sentinel_filter(defaults, file_cfg, cli_args):
    # Remove argparse sentinel None values before merging
    cli_filtered = {k: v for k, v in cli_args.items() if v is not None}
    return {**defaults, **file_cfg, **cli_filtered}

# Typical scenario: user set --lr 0.01 but left --batch_size unset (None)
defaults = {'lr': 0.001, 'batch_size': 32, 'epochs': 50, 'dropout': 0.1}
file_cfg = {'lr': 0.005, 'batch_size': 128}
cli_args = {'lr': 0.01, 'batch_size': None, 'epochs': None}  # batch_size unset

result = merge_with_sentinel_filter(defaults, file_cfg, cli_args)
print(f"lr:         {result['lr']}")         # 0.01  (CLI, explicitly set)
print(f"batch_size: {result['batch_size']}")  # 128   (file, CLI was None)
print(f"epochs:     {result['epochs']}")     # 50    (defaults, CLI was None)
print(f"dropout:    {result['dropout']}")    # 0.1   (defaults, absent from CLI)

assert result['lr'] == 0.01
assert result['batch_size'] == 128    # NOT overwritten by None
assert result['epochs'] == 50        # NOT overwritten by None
assert result['dropout'] == 0.1
print("Sentinel-None filter verified: unset CLI flags don't overwrite file config.")